# NB9 — V5 paired-ranking recheck

Controlled recheck of old S3.1 on top of the **current V5 regime**.

What stays fixed from V5:
- current `TypeAwarePairwiseScorer` architecture (`pair_hidden_dim=128`, `output_hidden_dim=16`);
- current category embedding init `N(0, 1/sqrt(32))` implemented by the model;
- BCE remains the main loss;
- FP32 train + FP32 validation;
- AdamW, LR `3e-4`, batch 256;
- max 60 epochs, patience 10, min 30;
- checkpoint selection by strict validation ROC-AUC;
- seed 42;
- test split is never loaded.

What changes experimentally:
- train batches keep each positive + its paired negative together;
- add `ranking_weight * softplus(-(positive_logit-negative_logit))`;
- try only weights `0.10` and `0.25`.

If the best candidate is close enough to V5 AUC, the final cell compares **pure LOO on original size >=4 only** against frozen V5. This avoids the 2-item extrapolation problem.


In [ ]:
from pathlib import Path
import copy
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "exp/v5-paired-ranking-recheck"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
print("Branch:", BRANCH)
print("Git HEAD:", HEAD)


In [ ]:
import yaml
import torch

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import (
    build_train_valid_loaders,
    evaluate_epoch,
    seed_everything,
    validate_s3_config,
)
from src.scorer.paired_ranking_experiment import (
    build_paired_train_loader,
    evaluate_pure_loo_4plus,
    fit_paired_ranking_scorer,
)

CONFIG_PATH = REPO_ROOT / "configs" / "scorer_v5_paired_ranking_recheck.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

validate_s3_config(config)
training = config["training"]
experiment = config["experiment"]

assert config["model"]["pair_hidden_dim"] == 128
assert config["model"]["output_hidden_dim"] == 16
assert training["mixed_precision"] is False
assert training["max_epochs"] == 60
assert training["early_stopping_patience"] == 10
assert training["early_stopping_min_epochs"] == 30
assert training["learning_rate"] == 0.0003
assert training["seed"] == 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime; this experiment is FP32."

paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
assert provenance["git_tree_clean"] is True

loaders = build_train_valid_loaders(paths, config, num_workers=0)
train_dataset = loaders["datasets"]["train"]
valid_dataset = loaders["datasets"]["valid"]
valid_loader = loaders["valid_loader"]

assert len(train_dataset) == 30918
assert len(valid_dataset) == 2284
assert len(train_dataset.pair_families) == 15459
assert len(valid_dataset.pair_families) == 1142

print("CONFIG / DATA / PROVENANCE: PASS")
print("Ranking weights:", experiment["ranking_weights"])
print("Device:", device)


In [ ]:
# Check that the current V5 initialization is really being used.
seed_everything(int(training["seed"]))
probe = TypeAwarePairwiseScorer.from_config(config)
with torch.no_grad():
    norms = probe.category_embedding.weight[1:].norm(dim=1)
print("category init policy:", probe.category_embedding_init_policy)
print("category init std:", probe.category_embedding_init_std)
print("mean category norm:", float(norms.mean()))
print("expected std 1/sqrt(32):", 32 ** -0.5)
del probe


In [ ]:
BASELINE_AUC = float(experiment["baseline_valid_roc_auc"])
BASELINE_FITB = float(experiment["baseline_valid_fitb_2way"])
RUN_ROOT = Path("/content/drive/MyDrive/v5_paired_ranking_recheck")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

results = []

for ranking_weight in [float(x) for x in experiment["ranking_weights"]]:
    print("\n" + "=" * 80)
    print(f"TRAINING ranking_weight={ranking_weight:.2f}")
    print("=" * 80)

    run_config = copy.deepcopy(config)
    run_config["experiment"]["active_ranking_weight"] = ranking_weight

    # Seed BEFORE model construction, then the fit helper reseeds training state.
    seed_everything(int(run_config["training"]["seed"]))
    model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
    paired_train_loader = build_paired_train_loader(
        train_dataset, run_config, num_workers=0
    )

    tag = f"rank_w_{ranking_weight:.2f}".replace(".", "p")
    run_dir = RUN_ROOT / f"{tag}_seed42"
    best_path = run_dir / "best.pt"

    if not best_path.is_file():
        fit_result = fit_paired_ranking_scorer(
            model,
            paired_train_loader,
            valid_loader,
            config=run_config,
            checkpoint_dir=run_dir,
            provenance=provenance,
            ranking_weight=ranking_weight,
            device=device,
        )
        print("TRAIN RESULT:", {
            "best_epoch": fit_result["best_epoch"],
            "best_valid_roc_auc": fit_result["best_valid_roc_auc"],
            "ranking_weight": fit_result["ranking_weight"],
        })
    else:
        print("Existing best.pt found; skipping retraining:", best_path)

    best_model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
    payload = load_checkpoint(
        best_path,
        model=best_model,
        map_location=device,
        current_provenance=provenance,
    )
    best_model.eval()

    criterion = torch.nn.BCEWithLogitsLoss()
    valid = evaluate_epoch(
        best_model,
        valid_loader,
        criterion=criterion,
        device=device,
    )

    row = {
        "ranking_weight": ranking_weight,
        "best_epoch": int(payload["epoch"]),
        "valid_roc_auc": float(valid["roc_auc"]),
        "valid_fitb_2way": float(valid["fitb_2way"]),
        "mean_logit_margin": float(valid["mean_logit_margin"]),
        "median_logit_margin": float(valid["median_logit_margin"]),
        "valid_loss": float(valid["loss"]),
        "delta_auc_vs_v5": float(valid["roc_auc"]) - BASELINE_AUC,
        "delta_fitb_vs_v5": float(valid["fitb_2way"]) - BASELINE_FITB,
        "best_path": str(best_path),
    }
    results.append(row)
    print("RESULT:", json.dumps(row, indent=2))

summary_path = RUN_ROOT / "paired_ranking_summary.json"
summary_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("\nSaved:", summary_path)
print("TEST SPLIT WAS NOT LOADED.")


In [ ]:
print("\nFINAL SCORER COMPARISON")
print(f"{'setting':>14} {'epoch':>6} {'AUC':>10} {'FITB':>10} {'dAUC':>10} {'dFITB':>10}")
print(f"{'V5 baseline':>14} {52:>6} {BASELINE_AUC:>10.6f} {BASELINE_FITB:>10.6f} {0.0:>+10.6f} {0.0:>+10.6f}")
for row in results:
    label = f"rank={row['ranking_weight']:.2f}"
    print(
        f"{label:>14} {row['best_epoch']:>6} "
        f"{row['valid_roc_auc']:>10.6f} {row['valid_fitb_2way']:>10.6f} "
        f"{row['delta_auc_vs_v5']:>+10.6f} {row['delta_fitb_vs_v5']:>+10.6f}"
    )

best_candidate = max(results, key=lambda r: r["valid_roc_auc"])
print("\nBest candidate by validation ROC-AUC:")
print(json.dumps(best_candidate, indent=2))


In [ ]:
# Downstream check: pure LOO on original-size >=4 only.
# This is intentionally not run on size-3 outfits because frozen/current V5
# accepts canonical inputs with >=3 items, and removing one item from n=3
# would create a 2-item extrapolation.

LOO_GATE_AUC_TOLERANCE = 0.002
RUN_LOO = best_candidate["valid_roc_auc"] >= BASELINE_AUC - LOO_GATE_AUC_TOLERANCE

if not RUN_LOO:
    print(
        "Skipping LOO: best ranking candidate is more than "
        f"{LOO_GATE_AUC_TOLERANCE:.3f} AUC below frozen V5."
    )
else:
    canonical_config_path = REPO_ROOT / "configs" / "scorer_type_aware_pairwise_v1_val_auc.yaml"
    with canonical_config_path.open("r", encoding="utf-8") as f:
        canonical_config = yaml.safe_load(f)

    frozen_path = (
        REPO_ROOT
        / "artifacts"
        / "checkpoints"
        / "type_aware_pairwise_v1"
        / "final_val_auc_v5_seed42"
        / "best.pt"
    )
    assert frozen_path.is_file(), frozen_path

    baseline_model = TypeAwarePairwiseScorer.from_config(canonical_config).to(device)
    load_checkpoint(
        frozen_path,
        model=baseline_model,
        map_location=device,
        current_provenance=provenance,
    )
    baseline_model.eval()

    candidate_model = TypeAwarePairwiseScorer.from_config(config).to(device)
    load_checkpoint(
        best_candidate["best_path"],
        model=candidate_model,
        map_location=device,
        current_provenance=provenance,
    )
    candidate_model.eval()

    print("\nRunning size>=4 pure LOO for frozen V5...")
    baseline_loo = evaluate_pure_loo_4plus(
        baseline_model, valid_dataset, device=device
    )

    print("Running size>=4 pure LOO for best paired-ranking candidate...")
    candidate_loo = evaluate_pure_loo_4plus(
        candidate_model, valid_dataset, device=device
    )

    loo_summary = {
        "scope": "validation negatives, original outfit size >=4 only",
        "baseline_v5": {
            k: v for k, v in baseline_loo.items() if k != "records"
        },
        "paired_ranking_candidate": {
            "ranking_weight": best_candidate["ranking_weight"],
            **{k: v for k, v in candidate_loo.items() if k != "records"},
        },
    }
    print("\nLOO COMPARISON")
    print(json.dumps(loo_summary, indent=2))

    loo_path = RUN_ROOT / "loo_4plus_comparison.json"
    loo_path.write_text(json.dumps(loo_summary, indent=2), encoding="utf-8")
    print("Saved:", loo_path)
    print("TEST SPLIT WAS NOT LOADED.")
